In [1]:
import numpy as np
from ase.io import read
from dscribe.descriptors import SOAP
from sklearn.decomposition import PCA
import chemiscope
from IPython.display import display

# 1. Load the structures
structures = read("dataset.xyz", index=":")

# 2. Setup SOAP (Smooth Overlap of Atomic Positions) descriptor
# We define the species present in your dataset (Carbon and Hydrogen)
species = ["C", "H"]

soap = SOAP(
    species=species,
    periodic=True,       # Your structures have periodic boundary conditions
    r_cut=5.0,            # Cutoff radius for local environment (in Angstroms)
    n_max=10,              # Number of radial basis functions
    l_max=4,              # Maximum degree of spherical harmonics
    average="inner",     # Averages the atomic environments into one global vector per structure
    sparse=False
)

print("Computing SOAP descriptors (this takes a moment)...")
# Create a matrix of shape (n_structures, n_features)
soap_features = soap.create(structures)

# 3. Dimensionality Reduction (PCA)
print("Running PCA to reduce features to 2D phase space...")
pca = PCA(n_components=3)
pca_embeddings = pca.fit_transform(soap_features)

# Extract PCA coordinates into lists for Chemiscope
pca1 = pca_embeddings[:, 0].tolist()
pca2 = pca_embeddings[:, 2].tolist()

# 4. Extract Physical Properties
properties = {
    "Structure_ID": list(range(len(structures))),
    "Energy": [s.get_potential_energy() for s in structures],
    "Max_Force": [abs(s.get_forces()).max() for s in structures],
    
    # Add our new Phase Space coordinates
    "PCA_1": pca1,
    "PCA_2": pca2
}

# 5. Visualize in Chemiscope
print("Launching Chemiscope...")
widget = chemiscope.show(
    structures=structures,
    properties=properties,
    settings=chemiscope.quick_settings(
        x="PCA_1",           # Map the X-axis to the first principal component
        y="PCA_2",           # Map the Y-axis to the second principal component
        map_color="Max_Force"   # Color the clusters by their potential energy
    )
)

display(widget)

Computing SOAP descriptors (this takes a moment)...
Running PCA to reduce features to 2D phase space...
Launching Chemiscope...


<ChemiscopeWidget(meta={'name': ' '}, settings={'map': {'joinPoints': False, 'x': {'property': 'PCA_1'}, 'y': …